In [2]:
from pathlib import Path
import torch
import pandas as pd

# reuse functions from your script
from main_predict import get_model, get_dataset, run_pred

/home/jovyan/work/MST/mst/models/extern/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/jovyan/work/MST/mst/models/extern/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/jovyan/work/MST/mst/models/extern/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

run_dir = Path("/home/jovyan/work/MST/runs")
run_folder = Path("ODELIA/DinoV2ClassifierSlice_Final")

path_run = run_dir / run_folder

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [5]:
ModelClass = get_model("DinoV2ClassifierSlice")

model = ModelClass.load_best_checkpoint(path_run)
model.to(device)
model.eval()


/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
/opt/conda/lib/python3.11/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEn

DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

In [6]:
ds_test = get_dataset(name="ODELIA", split="test")

In [7]:
sample = ds_test[0]  # first test case

In [8]:
batch = {}
for k, v in sample.items():
    if torch.is_tensor(v):
        batch[k] = v.unsqueeze(0).to(device)
    else:
        batch[k] = v


In [9]:
batch

{'uid': 'ODELIA_BRAID1_0246_1_left',
 'source': tensor([[[[[-0.3985, -0.4021, -0.3985,  ..., -0.4494, -0.4530, -0.4239],
            [-0.3949, -0.4094, -0.4094,  ..., -0.4385, -0.4385, -0.4058],
            [-0.4130, -0.4130, -0.4021,  ..., -0.3985, -0.4312, -0.4094],
            ...,
            [-0.4130, -0.4058, -0.3985,  ...,  1.0484,  1.0775,  0.9139],
            [-0.3912, -0.4021, -0.4457,  ...,  0.9794,  0.9794,  0.8412],
            [-0.3803, -0.3803, -0.4857,  ...,  0.5213,  0.4958,  0.5686]],
 
           [[-0.4058, -0.3985, -0.4167,  ..., -0.4312, -0.4167, -0.3658],
            [-0.4203, -0.4167, -0.4130,  ..., -0.4058, -0.3985, -0.3876],
            [-0.4058, -0.4021, -0.4021,  ..., -0.3803, -0.4058, -0.3985],
            ...,
            [-0.4094, -0.4167, -0.3985,  ...,  0.7140,  0.8303,  1.0303],
            [-0.3876, -0.4457, -0.4203,  ...,  0.7067,  0.7103,  0.7612],
            [-0.4239, -0.4312, -0.4167,  ...,  0.7249,  0.5504,  0.3977]],
 
           [[-0.4058, -0.

In [10]:
with torch.no_grad():
    pred, _, _ = run_pred(
        model,
        batch,
        save_attn=False,
        use_softmax=True,
        use_tta=False
    )

probs = torch.softmax(pred, dim=-1).cpu()
pred_class = torch.argmax(probs, dim=1)

print("GT:", batch["target"].item())
print("Predicted class:", pred_class.item())
print("Probabilities:", probs.squeeze().numpy())


GT: 0
Predicted class: 0
Probabilities: [0.57346475 0.21402398 0.21251123]
